# Recalibrate imbalance and optionally rebuild simulation bundle

This notebook is a reusable maintenance checkpoint for the BESS/MODO imbalance issue.

It recalibrates `data/processed/imbalance_params.json` from raw Elexon day-ahead Market Index and System Price data, using the defensive provider-level DA aggregation in `src.processes.imbalance.calibrate`. It can also rebuild `data/processed/sim_bundle.pkl` so notebooks 13 and 16 pick up the corrected imbalance process.

## Controls

Defaults are conservative: recalibration writes the corrected JSON, while the expensive bundle rebuild is switched off until you explicitly enable it.

In [1]:
from pathlib import Path
import json
import pickle
import shutil
import subprocess
import sys
from datetime import datetime

import numpy as np
import pandas as pd


def find_project_root(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / 'src').exists() and (p / 'data').exists():
            return p
    raise RuntimeError('Could not find project root containing src/ and data/.')


ROOT = find_project_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

RAW_DIR = ROOT / 'data' / 'raw'
PROC_DIR = ROOT / 'data' / 'processed'

DA_PATH = RAW_DIR / 'elexon_da_prices.parquet'
SP_PATH = RAW_DIR / 'elexon_sp_prices.parquet'
IMB_PARAMS_PATH = PROC_DIR / 'imbalance_params.json'
SIM_BUNDLE_PATH = PROC_DIR / 'sim_bundle.pkl'

# Recalibration controls
WRITE_IMBALANCE_PARAMS = True
CREATE_BACKUP = True
THRESHOLD_SIGMA = 2.5

# Bundle rebuild controls. Turn one of these on after checking the params.
RUN_SMOKE_BUNDLE = False
RUN_FULL_BUNDLE = False
SMOKE_PATHS = 1000
FULL_PATHS = 5000
N_STEPS = 17_520
SEED = 42

print(f'Project root: {ROOT}')
print(f'DA path:      {DA_PATH}')
print(f'SP path:      {SP_PATH}')
print(f'Params path:  {IMB_PARAMS_PATH}')
print(f'Bundle path:  {SIM_BUNDLE_PATH}')

Project root: G:\My Drive\Research\BESS\bess_project
DA path:      G:\My Drive\Research\BESS\bess_project\data\raw\elexon_da_prices.parquet
SP path:      G:\My Drive\Research\BESS\bess_project\data\raw\elexon_sp_prices.parquet
Params path:  G:\My Drive\Research\BESS\bess_project\data\processed\imbalance_params.json
Bundle path:  G:\My Drive\Research\BESS\bess_project\data\processed\sim_bundle.pkl


## Load and audit source data

The failure mode this catches is provider-level Market Index rows being treated as separate settlement periods. Zero-volume provider placeholders should not become real DA prices.

In [2]:
df_da = pd.read_parquet(DA_PATH)
df_sp = pd.read_parquet(SP_PATH)

print('DA rows:', len(df_da))
print('SP rows:', len(df_sp))
print('DA columns:', list(df_da.columns))
print('SP columns:', list(df_sp.columns))

keys = ['settlement_date', 'settlement_period']
print('\nDA unique settlement periods:', df_da[keys].drop_duplicates().shape[0])
print('SP unique settlement periods:', df_sp[keys].drop_duplicates().shape[0])

if 'data_provider' in df_da.columns:
    provider_audit = (
        df_da.assign(is_zero_volume=df_da.get('volume_mwh', pd.Series(np.nan, index=df_da.index)).fillna(0).eq(0))
        .groupby('data_provider')
        .agg(rows=('price_gbp_mwh', 'size'), zero_volume_rows=('is_zero_volume', 'sum'), mean_price=('price_gbp_mwh', 'mean'))
        .sort_values('rows', ascending=False)
    )
    display(provider_audit)
else:
    print('\nNo data_provider column found; DA may already be settlement-period aggregated.')

DA rows: 36229
SP rows: 36240
DA columns: ['settlement_date', 'settlement_period', 'price_gbp_mwh', 'volume_mwh', 'data_provider']
SP columns: ['settlement_date', 'settlement_period', 'system_sell_price', 'system_buy_price', 'net_imbalance_volume', 'system_price']

DA unique settlement periods: 36229
SP unique settlement periods: 36240


,rows,zero_volume_rows,mean_price
data_provider,,,
MID_VOLUME_WEIGHTED,36229,0,79.004273


## Calibrate imbalance process

This uses `src.processes.imbalance.calibrate`, which now aggregates DA rows to one positive-volume volume-weighted price per settlement period before merging with System Price.

In [3]:
from src.processes.imbalance import calibrate as calibrate_imbalance

old_params = None
if IMB_PARAMS_PATH.exists():
    old_params = json.loads(IMB_PARAMS_PATH.read_text())

imb_params = calibrate_imbalance(df_da, df_sp, dt=1.0, threshold_sigma=THRESHOLD_SIGMA)
new_params = imb_params.__dict__.copy()

summary = pd.DataFrame({'new': new_params})
if old_params:
    summary['old'] = pd.Series(old_params)
    summary['change'] = summary['new'] - summary['old']
display(summary)

stationary_jump_mean = (
    new_params['lambda_jump']
    * (new_params['p_pos'] * new_params['jump_scale_pos'] - (1 - new_params['p_pos']) * new_params['jump_scale_neg'])
    / new_params['theta_delta']
)
stationary_mean = new_params['mu_delta'] + stationary_jump_mean
print(f"\nImplied stationary mean including jumps: {stationary_mean:.3f} GBP/MWh")
print(f"Observations used: {new_params['n_obs']:,}")

,new,old,change
theta_delta,0.828093,0.828093,0.0
sigma_delta,18.650956,18.650956,0.0
lambda_jump,0.042866,0.042866,0.0
jump_scale_pos,109.758544,109.758544,0.0
jump_scale_neg,74.095006,74.095006,0.0
p_pos,0.358017,0.358017,0.0
mu_delta,0.220665,0.220665,0.0
log_likelihood,-213503.407221,-213503.407221,0.0
n_obs,36229.000000,36229.000000,0.0



Implied stationary mean including jumps: -0.208 GBP/MWh
Observations used: 36,229


## Write corrected params

When `WRITE_IMBALANCE_PARAMS=True`, the existing JSON is backed up before writing.

In [4]:
if WRITE_IMBALANCE_PARAMS:
    PROC_DIR.mkdir(parents=True, exist_ok=True)
    if CREATE_BACKUP and IMB_PARAMS_PATH.exists():
        stamp = datetime.now().strftime('%Y%m%d_%H%M%S')
        backup_path = IMB_PARAMS_PATH.with_name(f'imbalance_params_backup_{stamp}.json')
        shutil.copy2(IMB_PARAMS_PATH, backup_path)
        print(f'Backed up previous params to: {backup_path}')
    imb_params.to_json(IMB_PARAMS_PATH)
    print(f'Wrote corrected params to: {IMB_PARAMS_PATH}')
else:
    print('WRITE_IMBALANCE_PARAMS=False; no file was written.')

Backed up previous params to: G:\My Drive\Research\BESS\bess_project\data\processed\imbalance_params_backup_20260506_135046.json
Wrote corrected params to: G:\My Drive\Research\BESS\bess_project\data\processed\imbalance_params.json


## Optional: rebuild simulation bundle

Notebook 13 and notebook 16 read `data/processed/sim_bundle.pkl`. Rebuild it after changing imbalance params, otherwise the old simulated `delta_imb` paths remain in use.

Set `RUN_SMOKE_BUNDLE=True` for a quick 1,000-path test, then `RUN_FULL_BUNDLE=True` for the normal 5,000-path bundle.

In [5]:
def run_bundle(paths: int) -> None:
    cmd = [
        sys.executable,
        str(ROOT / 'scripts' / 'generate_sim_bundle.py'),
        '--paths', str(paths),
        '--steps', str(N_STEPS),
        '--seed', str(SEED),
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, cwd=ROOT, check=True)


if RUN_SMOKE_BUNDLE:
    run_bundle(SMOKE_PATHS)
if RUN_FULL_BUNDLE:
    run_bundle(FULL_PATHS)
if not RUN_SMOKE_BUNDLE and not RUN_FULL_BUNDLE:
    print('Bundle rebuild skipped. Enable RUN_SMOKE_BUNDLE or RUN_FULL_BUNDLE when ready.')

Bundle rebuild skipped. Enable RUN_SMOKE_BUNDLE or RUN_FULL_BUNDLE when ready.


## Optional: inspect current bundle

Use this after rebuilding to confirm the simulated imbalance paths are no longer dominated by a large positive basis.

In [6]:
if SIM_BUNDLE_PATH.exists():
    with SIM_BUNDLE_PATH.open('rb') as f:
        bundle = pickle.load(f)
    d = bundle.delta_imb
    print('paths, steps:', bundle.n_paths, bundle.n_steps)
    print('delta mean/std:', float(d.mean()), float(d.std()))
    print('delta p5/p50/p95:', np.percentile(d, [5, 50, 95]))
else:
    print(f'No bundle found at: {SIM_BUNDLE_PATH}')

paths, steps: 1000 17520
delta mean/std: -20.577495574951172 139.98846435546875
delta p5/p50/p95: [-239.60974579  -24.09610844  215.74279175]


## Next notebooks

After the full bundle rebuild, rerun:

1. `13_phase4_duration_sweep.ipynb` to update duration scaling and attribution.
2. `16_lsmc_counterfactual_sweep.ipynb` to confirm whether imbalance was the flatness driver.